In [0]:
# CELL 1 — Install ultralytics
%pip install ultralytics mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.7/419.7 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.6/196.6 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2

In [0]:
# CELL 2 — Restart Python
dbutils.library.restartPython()

In [0]:
# CELL 3 — Register model in MLflow without loading into memory
import mlflow
import os

# Verify the file exists in the Volume
model_path = "/Volumes/workspace/default/pothole_models/best.pt"
file_size = os.path.getsize(model_path) / (1024*1024)
print(f"✅ Model file found: {model_path}")
print(f"   File size: {file_size:.1f} MB")

✅ Model file found: /Volumes/workspace/default/pothole_models/best.pt
   File size: 6.0 MB


In [0]:
# CELL 4 — Log model metadata to MLflow (no loading needed)
mlflow.set_experiment("/pothole_heatmap_experiment")

with mlflow.start_run(run_name="yolov8_pothole_india_v1") as run:
    
    # Model info
    mlflow.log_param("model_architecture", "YOLOv8n")
    mlflow.log_param("dataset",            "BharatPotHole - 7000+ Indian dashcam frames")
    mlflow.log_param("training_platform",  "Kaggle Tesla T4 GPU")
    mlflow.log_param("epochs",             30)
    mlflow.log_param("image_size",         640)
    mlflow.log_param("classes",            "['pothole']")
    mlflow.log_param("train_split",        "70/20/10")
    
    # Metrics from Kaggle training
    mlflow.log_metric("mAP50",      0.392)
    mlflow.log_metric("mAP50_95",   0.146)
    mlflow.log_metric("precision",  0.497)
    mlflow.log_metric("recall",     0.421)
    
    # Log the weights file as an artifact
    mlflow.log_artifact(model_path, artifact_path="model_weights")
    
    run_id = run.info.run_id
    print(f"✅ Model logged to MLflow!")
    print(f"   Run ID:    {run_id}")
    print(f"   mAP50:     0.392")
    print(f"   Precision: 0.497")
    print(f"   Recall:    0.421")

2026/04/24 21:44:08 INFO mlflow.tracking.fluent: Experiment with name '/pothole_heatmap_experiment' does not exist. Creating a new experiment.


✅ Model logged to MLflow!
   Run ID:    8a778562879d4b918124b15ec844a23f
   mAP50:     0.392
   Precision: 0.497
   Recall:    0.421


In [0]:
# CELL 5 — Simulate batch inference results
# (YOLO ran on Kaggle GPU — we bring those results into Databricks)
import pandas as pd
import random
random.seed(42)

inference_records = []
cities = ["Chennai","Mumbai","Delhi","Bengaluru","Hyderabad","Kolkata"]

for i in range(200):
    city = random.choice(cities)
    has_pothole = random.random() < 0.15
    count = random.randint(1, 5) if has_pothole else 0
    conf  = round(random.uniform(0.45, 0.89), 3) if has_pothole else 0.0
    inference_records.append({
        "section_id":     random.randint(1, 50000),
        "city":           city,
        "image_path":     f"/dashcam/{city.lower()}/frame_{i:04d}.jpg",
        "pothole_count":  count,
        "avg_confidence": conf,
        "has_pothole":    1 if has_pothole else 0,
        "inference_ts":   pd.Timestamp.now().isoformat()
    })

inference_df = spark.createDataFrame(pd.DataFrame(inference_records))

inference_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pothole_heatmap.yolo_detections")

detected = sum(1 for r in inference_records if r["has_pothole"])
print(f"✅ Inference results saved to Delta Lake")
print(f"   Total sections processed: 200")
print(f"   Potholes detected: {detected}/200 sections")
inference_df.show(5)

✅ Inference results saved to Delta Lake
   Total sections processed: 200
   Potholes detected: 37/200 sections
+----------+-------+--------------------+-------------+--------------+-----------+--------------------+
|section_id|   city|          image_path|pothole_count|avg_confidence|has_pothole|        inference_ts|
+----------+-------+--------------------+-------------+--------------+-----------+--------------------+
|      9145|Kolkata|/dashcam/kolkata/...|            3|         0.558|          1|2026-04-24T21:44:...|
|     27652|Kolkata|/dashcam/kolkata/...|            5|         0.488|          1|2026-04-24T21:44:...|
|     39454|Chennai|/dashcam/chennai/...|            2|         0.552|          1|2026-04-24T21:44:...|
|     46926|Chennai|/dashcam/chennai/...|            0|           0.0|          0|2026-04-24T21:44:...|
|     27494|Kolkata|/dashcam/kolkata/...|            0|           0.0|          0|2026-04-24T21:44:...|
+----------+-------+--------------------+-------------+--

In [0]:
# CELL 6 — Test inference on a sample pothole image from the dataset
# Download a test image first
import urllib.request

test_img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3e/Pothole_at_MG_Road%2C_Pune.jpg/320px-Pothole_at_MG_Road%2C_Pune.jpg"
test_img_path = "/tmp/test_pothole.jpg"
urllib.request.urlretrieve(test_img_url, test_img_path)

# Run inference
results = yolo_model(test_img_path, verbose=False)
boxes = results[0].boxes
count = len(boxes) if boxes is not None else 0
conf  = float(boxes.conf.mean()) if count > 0 else 0.0

print(f"✅ Inference result on Pune road test image:")
print(f"   Potholes detected: {count}")
print(f"   Avg confidence:    {conf:.3f}")
print(f"   Has pothole:       {'YES 🔴' if count > 0 else 'NO 🟢'}")

In [0]:
# CELL 7 — Simulate batch inference and write results to Delta Lake
import pandas as pd
from pyspark.sql import functions as F

# Simulate 100 road section detections
# In production this would process real dashcam frames
import random
random.seed(42)

inference_records = []
for i in range(100):
    has_pothole = random.random() < 0.15   # 15% positive rate
    count = random.randint(1, 5) if has_pothole else 0
    conf  = round(random.uniform(0.45, 0.89), 3) if has_pothole else 0.0
    inference_records.append({
        "section_id":     random.randint(1, 50000),
        "image_path":     f"/dashcam/frame_{i:04d}.jpg",
        "pothole_count":  count,
        "avg_confidence": conf,
        "has_pothole":    1 if has_pothole else 0,
        "inference_ts":   pd.Timestamp.now().isoformat()
    })

inference_df = spark.createDataFrame(pd.DataFrame(inference_records))

inference_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pothole_heatmap.yolo_detections")

print(f"✅ Inference results saved to Delta Lake")
print(f"   Potholes found: {sum(1 for r in inference_records if r['has_pothole'])}/100 sections")
inference_df.show(5)

In [0]:
import mlflow
mlflow.set_experiment("/pothole_heatmap_experiment")

with mlflow.start_run(run_name="yolov8n_augmented_v2") as run:
    mlflow.log_param("model_architecture", "YOLOv8n")
    mlflow.log_param("epochs", 30)
    mlflow.log_param("augmentation", "True")
    mlflow.log_param("hsv_h", 0.015)
    mlflow.log_param("mosaic", 1.0)
    mlflow.log_metric("mAP50",     0.396)
    mlflow.log_metric("mAP50_95",  0.139)
    mlflow.log_metric("precision", 0.545)
    mlflow.log_metric("recall",    0.403)
    
    model_path = "/Volumes/workspace/default/pothole_models/best.pt"
    mlflow.log_artifact(model_path, artifact_path="model_weights")
    
    print(f"✅ Updated run logged: {run.info.run_id}")
    print(f"   Precision improved: 0.497 → 0.545 (+4.8%)")

✅ Updated run logged: ee21dd9f592b4ea6b154850a7ff832fe
   Precision improved: 0.497 → 0.545 (+4.8%)
